In [1]:
# python notebook for Make Your Own Neural Network
# code for a 3-layer neural network, and code for learning the MNIST dataset
# (c) Tariq Rashid, 2016
# license is GPLv2

In [2]:
import numpy
# scipy.special for the sigmoid function expit()
import scipy.special
# library for plotting arrays
import matplotlib.pyplot

In [3]:
# neural network class definition
class neuralNetwork:
    
    
    # initialise the neural network
    def __init__(self, inputnodes, hiddennodes, outputnodes, learningrate):
        # set number of nodes in each input, hidden, output layer
        self.inodes = inputnodes
        self.hnodes = hiddennodes
        self.onodes = outputnodes
        
        # link weight matrices, wih and who
        # weights inside the arrays are w_i_j, where link is from node i to node j in the next layer
        # w11 w21
        # w12 w22 etc 
        self.wih = numpy.random.normal(0.0, pow(self.inodes, -0.5), (self.hnodes, self.inodes))
        self.who = numpy.random.normal(0.0, pow(self.hnodes, -0.5), (self.onodes, self.hnodes))

        # learning rate
        self.lr = learningrate
        
        # activation function is the sigmoid function
        self.activation_function = lambda x: scipy.special.expit(x)
        
        pass

    
    # train the neural network
    def train(self, inputs_list, targets_list):
        # convert inputs list to 2d array
        inputs = numpy.array(inputs_list, ndmin=2).T
        targets = numpy.array(targets_list, ndmin=2).T
        
        # calculate signals into hidden layer
        hidden_inputs = numpy.dot(self.wih, inputs)
        # calculate the signals emerging from hidden layer
        hidden_outputs = self.activation_function(hidden_inputs)
        
        # calculate signals into final output layer
        final_inputs = numpy.dot(self.who, hidden_outputs)
        # calculate the signals emerging from final output layer
        final_outputs = self.activation_function(final_inputs)
        
        # output layer error is the (target - actual)
        output_errors = targets - final_outputs
        # hidden layer error is the output_errors, split by weights, recombined at hidden nodes
        hidden_errors = numpy.dot(self.who.T, output_errors) 
        
        # update the weights for the links between the hidden and output layers
        self.who += self.lr * numpy.dot((output_errors * final_outputs * (1.0 - final_outputs)), numpy.transpose(hidden_outputs))
        
        # update the weights for the links between the input and hidden layers
        self.wih += self.lr * numpy.dot((hidden_errors * hidden_outputs * (1.0 - hidden_outputs)), numpy.transpose(inputs))
        
        pass

    
    # query the neural network
    def query(self, inputs_list):
        # convert inputs list to 2d array
        inputs = numpy.array(inputs_list, ndmin=2).T
        
        # calculate signals into hidden layer
        hidden_inputs = numpy.dot(self.wih, inputs)
        # calculate the signals emerging from hidden layer
        hidden_outputs = self.activation_function(hidden_inputs)
        
        # calculate signals into final output layer
        final_inputs = numpy.dot(self.who, hidden_outputs)
        # calculate the signals emerging from final output layer
        final_outputs = self.activation_function(final_inputs)
        
        return final_outputs

In [4]:
# number of input, hidden and output nodes
input_nodes = 784
hidden_nodes = 200
output_nodes = 10

# learning rate
learning_rate = 0.1

# create instance of neural network
n = neuralNetwork(input_nodes,hidden_nodes,output_nodes, learning_rate)

In [5]:
# load MNIST training data from IDX files (full dataset)
import struct

def load_mnist_idx(images_path, labels_path):
    with open(labels_path, 'rb') as lbpath:
        magic, n_labels = struct.unpack('>II', lbpath.read(8))
        labels = numpy.frombuffer(lbpath.read(), dtype=numpy.uint8)

    with open(images_path, 'rb') as imgpath:
        magic, n_images, rows, cols = struct.unpack('>IIII', imgpath.read(16))
        images = numpy.frombuffer(imgpath.read(), dtype=numpy.uint8).reshape(n_images, rows * cols)

    if n_images != n_labels:
        raise ValueError('MNIST image/label count mismatch.')
    return images, labels

mnist_raw_dir = "../../../data/raw/MNIST/raw"
train_images, train_labels = load_mnist_idx(
    f"{mnist_raw_dir}/train-images-idx3-ubyte",
    f"{mnist_raw_dir}/train-labels-idx1-ubyte",
)
print("train samples =", train_images.shape[0])

train samples = 60000


In [6]:
# train the neural network

# epochs is the number of times the training data set is used for training
epochs = 5

for e in range(epochs):
    # go through all records in the training data set
    for i in range(train_images.shape[0]):
        # scale and shift the inputs
        inputs = (train_images[i].astype(numpy.float32) / 255.0 * 0.99) + 0.01
        # create the target output values (all 0.01, except the desired label which is 0.99)
        targets = numpy.zeros(output_nodes) + 0.01
        targets[int(train_labels[i])] = 0.99
        n.train(inputs, targets)
        pass
    print("finished epoch", e + 1, "of", epochs)
    pass

finished epoch 1 of 5
finished epoch 2 of 5
finished epoch 3 of 5
finished epoch 4 of 5
finished epoch 5 of 5


In [7]:
# load MNIST test data from IDX files (full dataset)
test_images, test_labels = load_mnist_idx(
    f"{mnist_raw_dir}/t10k-images-idx3-ubyte",
    f"{mnist_raw_dir}/t10k-labels-idx1-ubyte",
)
print("test samples =", test_images.shape[0])

test samples = 10000


In [8]:
# test the neural network

# scorecard for how well the network performs, initially empty
scorecard = []

# go through all the records in the test data set
for i in range(test_images.shape[0]):
    correct_label = int(test_labels[i])
    # scale and shift the inputs
    inputs = (test_images[i].astype(numpy.float32) / 255.0 * 0.99) + 0.01
    # query the network
    outputs = n.query(inputs)
    # the index of the highest value corresponds to the label
    label = numpy.argmax(outputs)
    # append correct or incorrect to list
    if (label == correct_label):
        scorecard.append(1)
    else:
        scorecard.append(0)
        pass
    
    pass

In [9]:
# calculate the performance score, the fraction of correct answers
scorecard_array = numpy.asarray(scorecard)
print ("performance = ", scorecard_array.sum() / scorecard_array.size)

performance =  0.974
